# Chapter 11: Pandas 

We saw in Chapter 9 how to import data from comma separated files (.csv) or other text files into numpy ndarrays with 1 or 2 axes, and how we can manipulate this data, add rows, columns etc.
One of the flaws of this approach is that it is really easy to make mistakes. While the data we read is structured (each row consisted of last name, first name, student id and several grades), we had to remember that midterm1 was a certain column, midterm2 a different one etc. These column numbers had no real meaning, and the order in which they were labelled was irrelevant. Basically, we have imported the *data* but lost the *meta-data*.

Maybe it would have been better to create a dictionary whose keys are the grade items and values list of grades:

In [ ]:
import numpy as np
grades  = np.loadtxt('grades.csv',delimiter = ',', skiprows= 1, usecols=(3, 4, 5, 6, 7, 8, 9), max_rows=10)


# Create a dictionary of lists from this data:
keys = ("Quizzes average", "HW1 mark", "HW2 mark", "HW3 mark", "Exam1 mark", "Exam2 mark", "Final exam mark")
gradesD = {}
for col, k in enumerate(keys):
    print(col, k)
    gradesD[k] = grades[:,col]
gradesD


This makes operations on columns really easy:

In [ ]:
gradesD['HW average'] = (gradesD['HW1 mark'] + gradesD['HW2 mark'] + gradesD['HW3 mark']) / 3

Operations on rows are a bit painful. Say we want all grades of Khan Vitaly with ID 7943944, we need to figure out that they are the 2nd students in the initial file

In [ ]:
row = 2
record = [gradesD[k][row] for k in gradesD.keys()]
print(record)

In retrospect, maybe we should have created a list of dictionaries:


In [ ]:
gradesL = []
for row in grades:
    record = {}
    for col, k in enumerate(keys):
        record[k] = row[col]
    gradesL.append(record)
print(gradesL)

But now, operation on columns are difficult, and even looking for a student is painful...

The problem is that while dictionary are good to represent data with key:value structure, they are not really designed to represent data whose structure is more complex.

This is where Pandas comes into play (note that there are other approaches, including numpy structured arrays).

## 11.1 Pandas `Series`
*Series* are another type of container that can store data of various type. Series can be many things, single data (int, str, ...), lists, nparrays, or dictionaries.


In [ ]:
import pandas as pd
rowL0 = pd.Series(grades[0,:])
rowL1 = pd.Series(grades[1,:])
rowL1
# print(rowL0[1])

In [ ]:
# print(gradesL[0])
rowD0 = pd.Series(gradesL[0])
rowD1 = pd.Series(gradesL[1])
print(rowD0)
print(rowD0['HW1 mark'])

Series can be converted to dictionaries, lists, numpy arrays etc. Just like ndarras, operations on Series are vectorized:

In [ ]:
print(rowD0.to_list())

In [ ]:
print(rowL1.to_dict())

In [ ]:
# pandas series can be indexed by key (XXX.loc[]) or index (XXX.iloc[])
print(rowD1.iloc[0])
print(rowD1.loc['Quizzes average'])

In [ ]:
print((rowD0 + rowD1)/2)

Pandas is actually pretty good at vectorizing operations, even when keys don't match exactly:

In [ ]:
rowD1['avg'] = 99
print(rowD1)
print((rowD0+rowD1)/2)

Finally, Series can be given a *name*. Here, maybe the student number could have been the name. Or maybe the students names could be the names. We'll see in a bit why. 

In [ ]:
print(rowD0.name)
rowD0.name = '620565656'
rowD1.name = '107856531'
# rowL0.name = '620565656'
# rowL1.name = '107856531'
print(rowD0.name)
print(rowD0)

## 11.2 Pandas `DataFrame`

A `DataFrame` is a 2-dimensional labeled data structure with columns of potentially different types. One way to think about a `Dataframe` is as a two-dimensional list of data, where both rows and columns are structured, like a spreadsheet, or a `dict` of `Series`.
You can create a `DataFrame` from a list or dict of Series, a 2 axes ndarray

In [ ]:
gradesDF = pd.DataFrame([rowD0, rowD1])
gradesDF

In [ ]:
print(gradesDF['HW1 mark'])
print(gradesDF.loc['620565656'])

Of course, pandas knows how to read a file directly into a DataFrame.

In [ ]:
gradesDF2 = pd.read_csv("grades.csv", skipinitialspace=True, index_col=2)
gradesDF2.head() # same as gradesDF2[:5]

We now have the best of a list of dictionaries and a dictionary of lists:

In [ ]:
print(type(gradesDF2['HW avg']))
print(type(gradesDF2.loc[7943944]))

In [ ]:
# selecting rows:

# loc is label-based, iloc is index-based
print(gradesDF2.loc[114037001])
print(gradesDF2.iloc[2])


In [ ]:
# selecting columns
print(gradesDF2['HW1 mark'][:10])


In [ ]:
# Operation on columns
gradesDF2['HW avg'] = (gradesDF2['HW1 mark'] + gradesDF2['HW2 mark'] + gradesDF2['HW3 mark']) / 3
gradesDF2.head()

## 11.3 Sorting

In [ ]:
gradesDF2.sort_values('HW avg', ascending=False).head()

## 11.4 Selecting / filtering

In [ ]:
A = gradesDF2['HW avg'] >= 90
gradesDF2[A]

In [ ]:
B = (gradesDF2['HW avg'] >= 85) & (gradesDF2['HW avg'] < 90)
gradesDF2[B]

In [ ]:
# Another way using a 'query'. Note the back quotes `
B2 = gradesDF2.query("80 <= `HW avg` < 90")
